# 🌾 Detectron2 Mask R-CNN — Farm Field Boundary Segmentation

| Item | Detail |
|------|--------|
| Task | Instance segmentation |
| Model | Mask R-CNN (ResNet-101-FPN) |
| License | **Apache 2.0** ✅ (commercial-safe) |
| Classes | 1 — `land-Sx1C` (farm/cropland boundaries) |
| Dataset | 62 images (43 train / 13 valid / 6 test), 640×640 |
| Framework | [Detectron2](https://github.com/facebookresearch/detectron2) (Meta FAIR) |

**Idempotent**: Safe to re-run — skips completed steps.

## 0️⃣ Environment Setup

In [ ]:
import os
os.environ['http_proxy']  = 'http://10.68.69.53:80/'
os.environ['https_proxy'] = 'http://10.68.69.53:80/'

import numpy as np
import torch
import json
from pathlib import Path

NOTEBOOK_DIR = Path('.').resolve()

print(f'PyTorch      : {torch.__version__}')
print(f'Notebook dir : {NOTEBOOK_DIR}')

if torch.cuda.is_available():
    print(f'CUDA         : {torch.cuda.get_device_name(0)}')
else:
    print(f'Device       : CPU only')

import detectron2
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_test_loader
from detectron2.data.datasets import register_coco_instances
from detectron2.utils.visualizer import Visualizer, ColorMode

print(f'Detectron2   : {detectron2.__version__}')

## 1️⃣ Convert YOLO → COCO Format

In [ ]:
import sys
sys.path.insert(0, str(NOTEBOOK_DIR))
from yolo_to_coco import yolo_to_coco

DATASET_DIR  = NOTEBOOK_DIR / 'Farm Fields.v1i.yolov8'
CLASS_NAMES  = ['land-Sx1C']

coco_jsons = yolo_to_coco(
    dataset_dir = str(DATASET_DIR),
    class_names = CLASS_NAMES,
    splits      = ['train', 'valid', 'test'],
    force       = False,
)

for split, path in coco_jsons.items():
    with open(path) as f:
        data = json.load(f)
    print(f'  {split}: {len(data["images"])} imgs, {len(data["annotations"])} anns')

## 2️⃣ Register Dataset with Detectron2

In [ ]:
DATASET_PREFIX = 'farm_fields'

for split, img_subdir in [('train', 'train/images'), ('val', 'valid/images'), ('test', 'test/images')]:
    ds_name = f'{DATASET_PREFIX}_{split}'
    json_key = {'train': 'train', 'val': 'valid', 'test': 'test'}[split]
    json_path = str(DATASET_DIR / f'annotations_{json_key}.json')
    img_dir   = str(DATASET_DIR / img_subdir)

    if ds_name in DatasetCatalog.list():
        DatasetCatalog.remove(ds_name)
        MetadataCatalog.remove(ds_name)

    register_coco_instances(ds_name, {}, json_path, img_dir)
    print(f'✅ Registered: {ds_name}')

metadata = MetadataCatalog.get(f'{DATASET_PREFIX}_train')
metadata.thing_classes = CLASS_NAMES
print(f'   Classes: {metadata.thing_classes}')

## 3️⃣ Sanity Check — Visualise GT Annotations

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

%matplotlib inline

train_dicts = DatasetCatalog.get(f'{DATASET_PREFIX}_train')
samples = random.sample(train_dicts, min(4, len(train_dicts)))

fig, axes = plt.subplots(1, len(samples), figsize=(20, 5))
for ax, d in zip(axes, samples):
    img = np.array(Image.open(d['file_name']))
    v = Visualizer(img, metadata=metadata, scale=1.0, instance_mode=ColorMode.SEGMENTATION)
    vis = v.draw_dataset_dict(d)
    ax.imshow(vis.get_image())
    ax.set_title(Path(d['file_name']).name[:20] + '…', fontsize=8)
    ax.axis('off')

fig.suptitle('Training samples with GT masks', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4️⃣ Configure Mask R-CNN

Using **R-101-FPN** backbone pretrained on COCO. Optimized for small dataset (43 images).

In [ ]:
OUTPUT_DIR = str(NOTEBOOK_DIR / 'output_detectron2')
os.makedirs(OUTPUT_DIR, exist_ok=True)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    'COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml'))

cfg.DATASETS.TRAIN = (f'{DATASET_PREFIX}_train',)
cfg.DATASETS.TEST  = (f'{DATASET_PREFIX}_val',)

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    'COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml')

cfg.SOLVER.IMS_PER_BATCH   = 2
cfg.SOLVER.BASE_LR         = 0.0005
cfg.SOLVER.MAX_ITER        = 10000
cfg.SOLVER.LR_SCHEDULER_NAME = 'WarmupCosineLR'
cfg.SOLVER.WARMUP_ITERS    = 500
cfg.SOLVER.WARMUP_METHOD   = 'linear'
cfg.SOLVER.WEIGHT_DECAY    = 0.0005
cfg.SOLVER.CHECKPOINT_PERIOD = 2000

cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASS_NAMES)
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3

cfg.DATALOADER.NUM_WORKERS = 2
cfg.INPUT.MIN_SIZE_TRAIN   = (480, 512, 544, 576, 608, 640)
cfg.INPUT.MAX_SIZE_TRAIN   = 800
cfg.INPUT.MIN_SIZE_TEST    = 640
cfg.INPUT.MAX_SIZE_TEST    = 800
cfg.INPUT.RANDOM_FLIP      = 'horizontal'

cfg.OUTPUT_DIR = OUTPUT_DIR
cfg.TEST.EVAL_PERIOD = 1000

print(f'✅ Config ready')
print(f'   Backbone    : R-101-FPN (COCO pretrained)')
print(f'   Max iters   : {cfg.SOLVER.MAX_ITER}')
print(f'   LR          : {cfg.SOLVER.BASE_LR}')
print(f'   Batch       : {cfg.SOLVER.IMS_PER_BATCH}')
print(f'   Output      : {OUTPUT_DIR}')

## 5️⃣ Save Pretrained (COCO) Predictions BEFORE Finetuning

In [ ]:
cfg_pretrained = cfg.clone()
cfg_pretrained.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg_pretrained.MODEL.ROI_HEADS.NUM_CLASSES = 80

test_dicts = DatasetCatalog.get(f'{DATASET_PREFIX}_test')

pretrained_plots = {}
try:
    predictor_coco = DefaultPredictor(cfg_pretrained)
    for d in test_dicts:
        img = np.array(Image.open(d['file_name']))
        outputs = predictor_coco(img[:, :, ::-1])  # RGB→BGR
        inst = outputs['instances'].to('cpu')
        v = Visualizer(img, scale=1.0, instance_mode=ColorMode.SEGMENTATION)
        # Masks only — no boxes
        if inst.has('pred_masks') and len(inst) > 0:
            vis = v.overlay_instances(masks=inst.pred_masks, alpha=0.5)
        else:
            vis = v.get_output()
        pretrained_plots[Path(d['file_name']).name] = vis.get_image()
    print(f'✅ Saved COCO-pretrained predictions for {len(pretrained_plots)} test images')
    del predictor_coco
    torch.cuda.empty_cache()
except Exception as e:
    print(f'⚠️  Could not run pretrained predictions: {e}')

## 6️⃣ Train Mask R-CNN

Skips if `model_final.pth` already exists.

In [ ]:
from detectron2.engine import hooks

final_model = Path(OUTPUT_DIR) / 'model_final.pth'

if final_model.exists():
    print(f'⏭️  Training already complete — {final_model} ({final_model.stat().st_size/1e6:.1f} MB)')
    print(f'   Delete {OUTPUT_DIR} to retrain.')
else:
    class TrainerWithEval(DefaultTrainer):
        @classmethod
        def build_evaluator(cls, cfg, dataset_name, output_folder=None):
            if output_folder is None:
                output_folder = os.path.join(cfg.OUTPUT_DIR, 'eval')
            return COCOEvaluator(dataset_name, output_dir=output_folder)

    trainer = TrainerWithEval(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
    print(f'\n✅ Training complete — {final_model}')

## 7️⃣ Training Curves

In [ ]:
metrics_file = Path(OUTPUT_DIR) / 'metrics.json'

if metrics_file.exists():
    metrics = []
    with open(metrics_file) as f:
        for line in f:
            metrics.append(json.loads(line.strip()))

    iters = [m['iteration'] for m in metrics if 'total_loss' in m]
    total_loss = [m['total_loss'] for m in metrics if 'total_loss' in m]
    mask_loss  = [m.get('loss_mask', 0) for m in metrics if 'total_loss' in m]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    ax1.plot(iters, total_loss, color='#FF6B6B', linewidth=0.8, alpha=0.5, label='raw')
    window = min(50, len(total_loss) // 5) or 1
    smoothed = np.convolve(total_loss, np.ones(window)/window, mode='valid')
    ax1.plot(iters[:len(smoothed)], smoothed, color='#FF6B6B', linewidth=2, label='smoothed')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('Total Loss')
    ax1.set_title('Total Loss', fontweight='bold'); ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(iters, mask_loss, color='#4ECDC4', linewidth=0.8, alpha=0.5, label='raw')
    smoothed_m = np.convolve(mask_loss, np.ones(window)/window, mode='valid')
    ax2.plot(iters[:len(smoothed_m)], smoothed_m, color='#4ECDC4', linewidth=2, label='smoothed')
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Mask Loss')
    ax2.set_title('Mask Loss', fontweight='bold'); ax2.legend(); ax2.grid(alpha=0.3)

    fig.suptitle('Training Curves', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  metrics.json not found')

## 8️⃣ Evaluate on Val & Test

In [ ]:
cfg.MODEL.WEIGHTS = str(final_model)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.3
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASS_NAMES)
predictor = DefaultPredictor(cfg)

for split_name in ['val', 'test']:
    ds_name = f'{DATASET_PREFIX}_{split_name}'
    evaluator = COCOEvaluator(ds_name, output_dir=os.path.join(OUTPUT_DIR, f'eval_{split_name}'))
    loader = build_detection_test_loader(cfg, ds_name)
    results = inference_on_dataset(predictor.model, loader, evaluator)

    print(f'\n{"="*50}')
    print(f'  {split_name.upper()} Results')
    print(f'{"="*50}')
    if 'segm' in results:
        seg = results['segm']
        print(f'  Segmentation:')
        print(f'    AP         : {seg["AP"]:.2f}')
        print(f'    AP50       : {seg["AP50"]:.2f}')
        print(f'    AP75       : {seg["AP75"]:.2f}')
    if 'bbox' in results:
        bbox = results['bbox']
        print(f'  BBox:')
        print(f'    AP         : {bbox["AP"]:.2f}')
        print(f'    AP50       : {bbox["AP50"]:.2f}')
    print(f'{"="*50}')

---

# 🔬 Comparisons

---

## 9️⃣ Ground Truth vs Mask R-CNN Predictions

**Masks only** — no bounding boxes.

In [ ]:
def draw_masks_only(img, instances, metadata):
    """Draw only segmentation masks on image — no boxes, no labels."""
    v = Visualizer(img, metadata=metadata, scale=1.0, instance_mode=ColorMode.SEGMENTATION)
    if instances.has('pred_masks') and len(instances) > 0:
        vis = v.overlay_instances(
            masks=instances.pred_masks,
            alpha=0.5,
        )
    else:
        vis = v.get_output()
    return vis.get_image()

In [ ]:
test_dicts = DatasetCatalog.get(f'{DATASET_PREFIX}_test')
n = len(test_dicts)

fig, axes = plt.subplots(n, 2, figsize=(16, 7 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for i, d in enumerate(test_dicts):
    img = np.array(Image.open(d['file_name']))

    # GT — masks only
    v_gt = Visualizer(img, metadata=metadata, scale=1.0, instance_mode=ColorMode.SEGMENTATION)
    vis_gt = v_gt.draw_dataset_dict(d)
    axes[i, 0].imshow(vis_gt.get_image())
    n_gt = len(d.get('annotations', []))
    axes[i, 0].set_title(f'Ground Truth — {n_gt} polygons', fontsize=11, fontweight='bold', color='lime')
    axes[i, 0].axis('off')

    # Prediction — masks only, no boxes
    outputs = predictor(img[:, :, ::-1])
    inst = outputs['instances'].to('cpu')
    pred_img = draw_masks_only(img, inst, metadata)
    n_pred = len(inst)
    axes[i, 1].imshow(pred_img)
    axes[i, 1].set_title(f'Mask R-CNN — {n_pred} fields', fontsize=11, fontweight='bold', color='cyan')
    axes[i, 1].axis('off')

fig.suptitle('A) Ground Truth vs Mask R-CNN Predictions', fontsize=15, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

## 🔟 Before vs After Finetuning

Left = **COCO pretrained** | Right = **Finetuned on farm fields**

In [ ]:
if not pretrained_plots:
    print('⏭️  Skipping — no pretrained predictions saved')
else:
    n = len(test_dicts)
    fig, axes = plt.subplots(n, 2, figsize=(16, 7 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i, d in enumerate(test_dicts):
        img = np.array(Image.open(d['file_name']))
        name = Path(d['file_name']).name

        # Before
        if name in pretrained_plots:
            axes[i, 0].imshow(pretrained_plots[name])
        else:
            axes[i, 0].imshow(img)
        axes[i, 0].set_title('BEFORE — COCO pretrained', fontsize=11, fontweight='bold', color='orange')
        axes[i, 0].axis('off')

        # After — masks only
        outputs = predictor(img[:, :, ::-1])
        inst = outputs['instances'].to('cpu')
        pred_img = draw_masks_only(img, inst, metadata)
        n_pred = len(inst)
        axes[i, 1].imshow(pred_img)
        axes[i, 1].set_title(f'AFTER — finetuned ({n_pred} fields)', fontsize=11, fontweight='bold', color='cyan')
        axes[i, 1].axis('off')

    fig.suptitle('B) Before vs After Finetuning', fontsize=15, fontweight='bold', y=1.005)
    plt.tight_layout()
    plt.show()

## 1️⃣1️⃣ Export Model

In [ ]:
from detectron2.export import TracingAdapter

onnx_path = Path(OUTPUT_DIR) / 'model.onnx'

if onnx_path.exists():
    print(f'⏭️  ONNX already exported: {onnx_path} ({onnx_path.stat().st_size/1e6:.1f} MB)')
else:
    standalone_path = Path(OUTPUT_DIR) / 'model_standalone.pth'
    if not standalone_path.exists():
        torch.save(predictor.model.state_dict(), str(standalone_path))
        print(f'✅ Saved standalone weights: {standalone_path} ({standalone_path.stat().st_size/1e6:.1f} MB)')

    try:
        sample_input = torch.randn(1, 3, 640, 640).to(predictor.model.device)
        wrapper = TracingAdapter(predictor.model, sample_input)
        torch.onnx.export(wrapper, sample_input, str(onnx_path), opset_version=11)
        print(f'✅ ONNX exported: {onnx_path}')
    except Exception as e:
        print(f'⚠️  ONNX export not supported for Mask R-CNN: {e}')
        print(f'   Use model_standalone.pth or model_final.pth for deployment')

print(f'\n📁 All outputs: {Path(OUTPUT_DIR).resolve()}')
print(f'   Final weights : {final_model}')

## 1️⃣2️⃣ Inference on Custom Image

Field boundaries only — no bounding boxes.

In [ ]:
CUSTOM_IMAGE = Path('/mnt/raid1-backup/houcine/field_delineation_yolo/75_jpg.rf.f423d98d6d6ce09dfedef22e658ccec8.jpg')

assert CUSTOM_IMAGE.exists(), f'Image not found: {CUSTOM_IMAGE}'

img = np.array(Image.open(CUSTOM_IMAGE))
outputs = predictor(img[:, :, ::-1])
inst = outputs['instances'].to('cpu')
n_instances = len(inst)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

ax1.imshow(img)
ax1.set_title('Original', fontsize=12, fontweight='bold')
ax1.axis('off')

pred_img = draw_masks_only(img, inst, metadata)
ax2.imshow(pred_img)
ax2.set_title(f'Mask R-CNN — {n_instances} fields delineated', fontsize=12, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
if len(inst) > 0:
    print(f'\n🔎 {len(inst)} fields detected:')
    for i in range(len(inst)):
        score = float(inst.scores[i])
        mask = inst.pred_masks[i].numpy()
        area_px = int(mask.sum())
        print(f'   Field {i+1}: conf={score:.2f}  area={area_px} px')
else:
    print('No fields detected. Try lowering SCORE_THRESH_TEST.')

---

# 🌍 GeoTIFF Inference Pipeline

---

## 1️⃣3️⃣ GeoTIFF Inference → GeoJSON Export

Run Mask R-CNN on a real **GeoTIFF** satellite image:

1. Tile the raster into 640×640 windows
2. Run inference on each tile
3. Vectorise predicted masks → polygons in **geographic coordinates**
4. Export as **GeoJSON** (loadable in QGIS, Kepler.gl, geojson.io)

> ⚠️ Requires `rasterio` and `shapely` (both Apache 2.0).

In [ ]:
# ── §13  GeoTIFF Inference → GeoJSON Export ────────────────────
#       RFC 7946 compliant — WGS84 coords, no 'crs' field
#       ✅ Compatible with Kepler.gl, geojson.io, QGIS

import rasterio
import rasterio.features
import rasterio.windows
from shapely.geometry import shape, mapping
from shapely.ops import unary_union, transform as shapely_transform
from shapely.validation import make_valid
from pyproj import Transformer
import cv2

# ── USER CONFIG ────────────────────────────────────────────────
GEOTIFF_PATH = Path('/mnt/raid1-backup/houcine/field_delineation_yolo/sample_field.tif')  # ← change me
TILE_SIZE    = 640          # must match training resolution
OVERLAP      = 64           # pixels of overlap between tiles
CONF_THRESH  = 0.3          # minimum confidence to keep
GEOJSON_OUT  = Path(OUTPUT_DIR) / 'field_boundaries.geojson'
# ──────────────────────────────────────────────────────────────

assert GEOTIFF_PATH.exists(), f'GeoTIFF not found: {GEOTIFF_PATH}'

def geotiff_to_geojson(
    tif_path: Path,
    predictor,
    tile_size: int = TILE_SIZE,
    overlap: int = OVERLAP,
    conf_thresh: float = CONF_THRESH,
) -> dict:
    """
    Run Mask R-CNN on a GeoTIFF and return a RFC 7946 GeoJSON FeatureCollection.

    All coordinates are reprojected to EPSG:4326 (WGS84) for
    compatibility with Kepler.gl, geojson.io, and other web tools.
    Area (m²) is computed in the raster's native CRS before reprojection.
    """
    features = []
    step = tile_size - overlap

    with rasterio.open(tif_path) as src:
        src_crs = src.crs
        transform = src.transform
        img_h, img_w = src.height, src.width
        n_bands = src.count

        # Build reprojection transformer: native CRS → WGS84
        need_reproject = not src_crs.to_epsg() == 4326
        if need_reproject:
            transformer = Transformer.from_crs(src_crs, 'EPSG:4326', always_xy=True)
            reproject_fn = lambda x, y: transformer.transform(x, y)
            print(f'🔄 Will reproject: {src_crs} → EPSG:4326 (WGS84)')
        else:
            print(f'✅ Raster already in EPSG:4326 — no reprojection needed')

        print(f'📐 Raster : {img_w}×{img_h} px, {n_bands} bands')
        print(f'🌐 CRS    : {src_crs}')
        print(f'📏 Res    : {abs(transform.a):.2f} × {abs(transform.e):.2f} units/px')

        # Count tiles
        cols = max(1, (img_w - overlap) // step + (1 if (img_w - overlap) % step else 0))
        rows = max(1, (img_h - overlap) // step + (1 if (img_h - overlap) % step else 0))
        total_tiles = rows * cols
        print(f'🔲 Tiles  : {rows}×{cols} = {total_tiles}')

        tile_idx = 0
        for y0 in range(0, img_h, step):
            for x0 in range(0, img_w, step):
                # Clamp to image bounds
                x1 = min(x0 + tile_size, img_w)
                y1 = min(y0 + tile_size, img_h)
                win_w = x1 - x0
                win_h = y1 - y0

                if win_w < 32 or win_h < 32:
                    continue

                # Read tile
                window = rasterio.windows.Window(x0, y0, win_w, win_h)
                tile = src.read(window=window)  # (bands, h, w)

                # Prepare for Detectron2: needs BGR uint8 HWC
                if n_bands >= 3:
                    rgb = np.stack([tile[0], tile[1], tile[2]], axis=-1)  # HWC RGB
                else:
                    rgb = np.stack([tile[0]] * 3, axis=-1)

                # Ensure uint8
                if rgb.dtype != np.uint8:
                    if rgb.max() > 255:
                        rgb = np.clip(rgb / rgb.max() * 255, 0, 255).astype(np.uint8)
                    else:
                        rgb = rgb.astype(np.uint8)

                bgr = rgb[:, :, ::-1]  # RGB → BGR for Detectron2

                # Pad to tile_size if the tile is smaller (edge tiles)
                if win_h < tile_size or win_w < tile_size:
                    padded = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
                    padded[:win_h, :win_w] = bgr
                    bgr = padded

                # Inference
                outputs = predictor(bgr)
                instances = outputs['instances'].to('cpu')

                # Filter by confidence
                keep = instances.scores >= conf_thresh
                instances = instances[keep]

                if len(instances) == 0:
                    tile_idx += 1
                    continue

                # Vectorise each mask
                for i in range(len(instances)):
                    mask = instances.pred_masks[i].numpy().astype(np.uint8)
                    score = float(instances.scores[i])

                    # Crop mask to actual tile dimensions (remove padding)
                    mask_crop = mask[:win_h, :win_w]

                    # Vectorise mask → native-CRS polygons
                    tile_transform = rasterio.transform.from_bounds(
                        *rasterio.windows.bounds(window, transform),
                        win_w, win_h,
                    )

                    for geom, val in rasterio.features.shapes(
                        mask_crop, mask=mask_crop, transform=tile_transform
                    ):
                        if val == 0:
                            continue
                        poly = shape(geom)
                        if not poly.is_valid:
                            poly = make_valid(poly)
                        if poly.is_empty:
                            continue

                        # Compute area in native CRS (m² if projected)
                        area_native = poly.area

                        # Reproject to WGS84 for Kepler.gl
                        if need_reproject:
                            poly = shapely_transform(reproject_fn, poly)

                        features.append({
                            'type': 'Feature',
                            'properties': {
                                'confidence': round(score, 4),
                                'area_m2': round(area_native, 2),
                                'tile_idx': tile_idx,
                                'class': 'land-Sx1C',
                            },
                            'geometry': mapping(poly),
                        })

                tile_idx += 1
                if tile_idx % 50 == 0:
                    print(f'   Processed {tile_idx}/{total_tiles} tiles … ({len(features)} polygons so far)')

    print(f'\n✅ Done: {tile_idx} tiles processed, {len(features)} field polygons extracted')

    # RFC 7946 GeoJSON — no 'crs' field, coordinates in WGS84
    return {
        'type': 'FeatureCollection',
        'features': features,
    }


# ── Run inference ─────────────────────────────────────────────
geojson_fc = geotiff_to_geojson(GEOTIFF_PATH, predictor)

# ── Write GeoJSON ─────────────────────────────────────────────
with open(GEOJSON_OUT, 'w') as f:
    json.dump(geojson_fc, f)

n_fields = len(geojson_fc['features'])
size_kb = GEOJSON_OUT.stat().st_size / 1024
print(f'\n📄 GeoJSON : {GEOJSON_OUT}')
print(f'   Fields  : {n_fields}')
print(f'   Size    : {size_kb:.0f} KB')

if n_fields > 0:
    areas = [f['properties']['area_m2'] for f in geojson_fc['features']]
    confs = [f['properties']['confidence'] for f in geojson_fc['features']]
    print(f'   Area    : {np.median(areas):.0f} m² (median), {sum(areas):.0f} m² (total)')
    print(f'   Conf    : {np.mean(confs):.2f} (mean), {np.min(confs):.2f}–{np.max(confs):.2f} (range)')
print(f'\n🌐 Open in: https://kepler.gl/demo  or  https://geojson.io  or  QGIS')

## 1️⃣4️⃣ Visualise GeoJSON Results

Overlay predicted field boundaries on the GeoTIFF raster.
Polygons are coloured by **confidence score**.

In [ ]:
# ── §14  Visualise GeoJSON Results ─────────────────────────────

from matplotlib.collections import PatchCollection
from matplotlib.colors import Normalize
import matplotlib.cm as cm

def visualise_geojson_on_raster(
    tif_path: Path,
    geojson_data: dict,
    figsize=(18, 14),
    max_display: int = 5000,
):
    """
    Plot field boundaries from GeoJSON overlaid on the GeoTIFF raster.
    Polygons are coloured by confidence score.
    """
    features = geojson_data['features'][:max_display]
    if not features:
        print('⚠️  No features to display')
        return

    # Read raster for background
    with rasterio.open(tif_path) as src:
        # Read RGB bands (or first band ×3)
        if src.count >= 3:
            rgb = np.stack([src.read(b) for b in [1, 2, 3]], axis=-1)
        else:
            band = src.read(1)
            rgb = np.stack([band] * 3, axis=-1)

        # Normalise to 0–255 for display
        if rgb.dtype != np.uint8:
            p2, p98 = np.percentile(rgb[rgb > 0], [2, 98]) if rgb.max() > 0 else (0, 1)
            rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6) * 255, 0, 255).astype(np.uint8)

        extent = [
            src.bounds.left, src.bounds.right,
            src.bounds.bottom, src.bounds.top,
        ]
        crs = src.crs

    # ── Figure ────────────────────────────────────────────────
    fig, (ax_main, ax_hist) = plt.subplots(
        1, 2, figsize=figsize,
        gridspec_kw={'width_ratios': [3, 1]},
    )

    # Raster background
    ax_main.imshow(rgb, extent=extent, origin='upper', aspect='equal')

    # Collect patches + colours
    patches = []
    confs = []
    for feat in features:
        geom = shape(feat['geometry'])
        conf = feat['properties']['confidence']
        confs.append(conf)

        if geom.geom_type == 'Polygon':
            coords = np.array(geom.exterior.coords)
            patches.append(plt.Polygon(coords, closed=True))
        elif geom.geom_type == 'MultiPolygon':
            for part in geom.geoms:
                coords = np.array(part.exterior.coords)
                patches.append(plt.Polygon(coords, closed=True))
                confs.append(conf)

    # Colour by confidence
    norm = Normalize(vmin=CONF_THRESH, vmax=1.0)
    cmap = cm.get_cmap('RdYlGn')  # red=low → green=high
    colours = [cmap(norm(c)) for c in confs[:len(patches)]]

    pc = PatchCollection(
        patches, facecolors=colours,
        edgecolors=[(r, g, b, 1.0) for r, g, b, _ in colours],
        linewidths=0.6, alpha=0.55,
    )
    ax_main.add_collection(pc)

    # Colorbar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax_main, fraction=0.025, pad=0.02)
    cbar.set_label('Confidence', fontsize=11)

    ax_main.set_title(
        f'Mask R-CNN Field Delineation — {len(features)} fields',
        fontsize=14, fontweight='bold',
    )
    ax_main.set_xlabel('Easting' if crs.is_projected else 'Longitude', fontsize=10)
    ax_main.set_ylabel('Northing' if crs.is_projected else 'Latitude', fontsize=10)

    # ── Histogram of areas ────────────────────────────────────
    areas = [f['properties']['area_m2'] for f in features]
    conf_vals = [f['properties']['confidence'] for f in features]

    ax_hist.hist(areas, bins=30, color='#4ECDC4', edgecolor='white', alpha=0.85)
    ax_hist.set_xlabel('Area (m²)', fontsize=10)
    ax_hist.set_ylabel('Count', fontsize=10)
    ax_hist.set_title('Field Area Distribution', fontsize=12, fontweight='bold')
    ax_hist.grid(axis='y', alpha=0.3)

    # ── Summary text ──────────────────────────────────────────
    summary = (
        f'Fields: {len(features)}\n'
        f'Median area: {np.median(areas):.0f} m²\n'
        f'Total area: {sum(areas):,.0f} m²\n'
        f'Mean conf: {np.mean(conf_vals):.2f}\n'
        f'CRS: {crs}'
    )
    ax_hist.text(
        0.95, 0.95, summary,
        transform=ax_hist.transAxes,
        fontsize=9, verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8),
    )

    plt.tight_layout()
    # Save figure
    fig_path = Path(OUTPUT_DIR) / 'field_boundaries_map.png'
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'💾 Saved figure: {fig_path}')
    plt.show()


# ── Run visualisation ─────────────────────────────────────────
visualise_geojson_on_raster(GEOTIFF_PATH, geojson_fc)
print(f'\n🎉 Pipeline complete!')
print(f'   GeoJSON  : {GEOJSON_OUT}')
print(f'   Figure   : {Path(OUTPUT_DIR) / "field_boundaries_map.png"}')